country isn't updated anymore so we are using an old year 

In [1]:
platformID = 'TWI'

In [2]:
from datetime import datetime
import pandas as pd

import psycopg2


## import helper

In [3]:
import sys
from pathlib import Path

try:
    # Works in Python scripts
    helper_path = Path(__file__).resolve().parent.parent / "helper"
except NameError:
    # Works in Jupyter notebooks
    helper_path = Path().resolve().parent / "helper"

sys.path.insert(0, str(helper_path))

# Now import your modules 
from config import gam_info

from functions import execute_sql_query
import test_functions

In [4]:
platformID

'TWI'

In [5]:
# country
country_cols = ['PlaceID', 'TWI_CountryName']
country_codes = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='CountryID', 
                              keep_default_na=False)[country_cols]

# week 
week_tester = pd.read_excel(f"../../{gam_info['lookup_file']}", sheet_name='GAM Period')
week_tester['w/c'] = pd.to_datetime(week_tester['w/c'])

# social media accounts
dtype_dict = {'Channel ID': 'str',
              'Linked FB Account': 'str'}
socialmedia_accounts = pd.read_excel(f"../../{gam_info['lookup_file']}", dtype=dtype_dict,
                                     sheet_name='Social Media Accounts new')
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['PlatformID'] == platformID]
socialmedia_accounts = socialmedia_accounts[socialmedia_accounts['Status'] == 'active']
socialmedia_accounts['Channel ID'] = socialmedia_accounts['Channel ID'].dropna().apply(lambda x: str(int(x)))

channel_ids = socialmedia_accounts['Channel ID'].unique().tolist()
formatted_channel_ids = ', '.join(f"'{channel_id}'" for channel_id in channel_ids)

### RUN TESTS
test_functions.test_lookup_files(country_codes, country_cols, [f"{platformID}_country_0", f"{platformID}_country_1", f"{platformID}_country_2"], 
                                 test_step="lookup files - ensuring country codes is correct")

test_functions.test_lookup_files(week_tester, ['w/c'], [f"{platformID}_country_3", f"{platformID}_country_4", f"{platformID}_country_5"], 
                                 test_step = "lookup files - ensuring week tester is correct")

test_functions.test_lookup_files(socialmedia_accounts, ['Channel ID'], [f"{platformID}_country_6", f"{platformID}_country_7", f"{platformID}_country_8"], 
                                 test_step = "lookup files - ensuring social media accounts is correct")


✅ Test TWI_country_0 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TWI_country_1 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TWI_country_2 passed: No missing values in lookup.
...updating logbook...

✅ Test TWI_country_3 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TWI_country_4 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TWI_country_5 passed: No missing values in lookup.
...updating logbook...

✅ Test TWI_country_6 passed: lookup DataFrame is not empty.
...updating logbook...

✅ Test TWI_country_7 passed: No combinations occurs more than once a week.
...updating logbook...

✅ Test TWI_country_8 passed: No missing values in lookup.
...updating logbook...



# ingestion

# country

In [6]:
# takes forever to load
cols = ['Week Number', 'TW Account ID', 'Country', 'Weekly Engagements', 'Engagement %', 'Weekly Video Views']
tw_country_raw = pd.read_excel(f'../data/stale/stale_Twitter Engagements inc country.xlsx')[cols]
# ensure the accounts are strings (TW & FB)
tw_country_raw['TW Account ID'] = tw_country_raw['TW Account ID'].astype(str)

tw_country_raw = tw_country_raw.rename(columns={"TW Account ID": "Channel ID", 
                                              "Week Number": "WeekNumber_finYear",
                                               'Country': 'TWI_CountryName'})

In [7]:
tw_country_raw['TWI_CountryName'] = tw_country_raw['TWI_CountryName'].replace("Other", "Unknown").fillna('Unknown')

tw_country_df = tw_country_raw.merge(country_codes, on='TWI_CountryName', how='left')
test_functions.test_inner_join(tw_country_raw, country_codes, 
                               ['TWI_CountryName'], 
                               f"{platformID}_country_9", 
                               test_step='checking country in lookup',
                               focus='left')

# add service 
tw_country_df = tw_country_df.merge(socialmedia_accounts[['Channel ID', 'ServiceID', 
                                                          'Excluding UK', 'Linked FB Account']], 
                                    on='Channel ID', how='left')
test_functions.test_inner_join(tw_country_df, socialmedia_accounts[['Channel ID', 'ServiceID',
                                                          'Excluding UK', 'Linked FB Account']], 
                               ['Channel ID'], 
                               f"{platformID}_country_11", 
                               test_step='checking service join in lookup',
                               focus='left')

# test accounts 
column_name = 'Channel ID'
test_functions.test_filter_elements_returned(tw_country_df, channel_ids, column_name, 
                                             f'{platformID}_country_10', 
                                             test_step= 'stale country dataset - channels')

✅ Inner join test TWI_country_9 successful: No issues found.
...updating logbook...

✅ Inner join test TWI_country_11 successful: No issues found.
...updating logbook...

...testing Channel ID...
❌ Test TWI_country_10 failed: not all elements from lookup were retrieved in dataset - decide if they are really missing or if these accounts are inactive 
...updating logbook...



In [8]:
tw_country_df.to_csv(f"../data/processed/{platformID}/{gam_info['file_timeinfo']}_{platformID}_country.csv", 
                     index=None, na_rep='')

In [10]:
tw_country_df['Channel ID'].unique()

array(['18186782', '1566180572', '10012122', '18168998', '37740489',
       '245732153', '146478129', '146026726', '36670025', '1410692299',
       '209110566', '71229689', '159123210', '23772644', '790728',
       '826091183491387394', '18308305', '132849881', '791197',
       '19389095', '397349361', '2363005214', '18186609', '819192906',
       '4417219703', '85626267', '786764', '826099053788327936',
       '529467983', '978020802', '826117446050381826', '2953131377',
       '3422132321', '52032722', '1037252171472023553',
       '826107626807238657', '2493238856', '17994556', '706876069',
       '442782547', '4503599650261817', '87584044', '1207654748',
       '742143', '18164632', '2613311137', '92806602', '3646448592',
       '841652087641538560', '2450291', '1072811774569664512',
       '4339015883', '233996626', '22475139', '52344859', '18186700',
       '1392163634', '70725281', '397993663', '197442453',
       '1192362990063931394', '5402612', '23765365', '410605574',
      